# Machine Learning for Membrane Development

A worked example of machine learning (ML) on a table of **membrane** experiments: how a thin film was made, how well water and salt went through it, and how we might *suggest* a better recipe.

**Machine learning** here means a computer finds patterns in past experiments so it can guess the result of a *new* recipe. It is not a physics simulator, and it does not understand chemistry.

| You need | You do not need |
|----------|-----------------|
| Basic Python (variables, functions, and running cells **top to bottom**) | A membrane course, or prior experience with optimization |

The review article uses a **nine-step closed loop**: decide the goal, prepare the table, fit a model, suggest recipes, test them in the lab, and add the new results so the next suggestion is better. This notebook **discusses all nine steps** in that order.

You **run modeling code for Steps 4-7**. Steps 1-3 are decisions you make before a model is fit. The notebook can load and check a table that is already prepared, but it does not choose the scientific goal for you. Steps 8-9 are laboratory work. They are discussed here and are **not** automated.

| Paper step | In this notebook |
|------------|------------------|
| 1 Set the goal | Discussion. The goal is already written in `config.yaml`. |
| 2 Design the table | Discussion, then load a table that already follows the rules. |
| 3 Inspect the rows | Discussion, then count blanks and drop exact duplicate rows. |
| 4 Add columns and plot | Add ion size and charge, then look at distributions and a correlation heatmap. |
| 5 Fit and test a model | ExtraTrees, a train/test split, and a parity plot. |
| 6 Check the drivers (SHAP) | See which inputs the model used before you trust a search. |
| 7 Search for recipes | Ax prints a shortlist. It does not make a membrane. |
| 8 Test in the lab | Discussion only: make a few films and measure them. |
| 9 Update the model | Discussion only: add the new rows and run the modeling steps again. |

## Learning objectives

1. Write down what "good" means, and what the lab cannot change, before fitting a model.
2. Tell a **controllable setting** (something you can change while making the film) from a **fixed descriptor** (a number that comes with a chosen chemical) and from a **measurement made after the film exists**.
3. Tell **features** (inputs) from **targets** (numbers you want to predict).
4. Load a table and decide what to do with blank cells and duplicate rows.
5. Add chemistry columns, such as ion size and charge, *before* training, then look at the data.
6. Hold back a test set, fit a tree model, and read a **parity plot**.
7. Read **SHAP** plots as a check, not as proof of a chemical mechanism.
8. Run a short recipe search and turn the printout into instructions a lab could follow.

### Two kinds of prediction

- **Regression:** the target is a number, such as how fast water passes or how much salt is rejected. This tutorial predicts numbers.
- **Classification:** the target is a label, such as `pass` or `fail`. Other examples in this repository do that. The recipe search in Step 7 is **not** used for labels.

A high test score means "the model matched rows it had not seen." It does **not** mean "we discovered the true physics."

On **Google Colab**, run the bootstrap cell below **before** Setup.


### Prerequisites and setup (Google Colab)

Opening this `.ipynb` from GitHub does **not** download `config.yaml`, `src/`, or `data/`.

> **Next cell:** clone/update the repo and `pip install -e ".[all]"` (core + SHAP + Ax). First time can take several minutes. No Google Drive.

**Repository (repo):** the folder that contains `config.yaml`, `src/`, and `data/`.  
**`-e`:** editable install — imports come from `src/` on disk.

### How to read the output

- `Repo root: /content/ML-MEM-Review`
- `Colab bootstrap done.`

---

**Watch out.** Skipping this cell → missing `config.yaml` or `No module named utl_...`. After a GitHub push, run bootstrap again.  
**Local:** from the repo root, once, in *this* kernel: `pip install -e ".[all]"`. Notes: [`docs/COLAB.md`](../../docs/COLAB.md).


In [ ]:
# @title Bootstrap (clone + install)
# === COLAB BOOTSTRAP (safe locally) =========================================
# Clones Oikiy/ML-MEM-Review if you only opened the .ipynb from GitHub,
# installs the package editable, and cd's to the repo root.
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
REPO_REF = os.environ.get("ML_MEM_REPO_REF", "main")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))
EXTRAS = 'all'

def _git_head(repo: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return ""

def _truthy(name: str) -> bool:
    return os.environ.get(name, "").strip().lower() in {"1", "true", "yes"}

if IN_COLAB:
    if COLAB_ROOT.exists() and (COLAB_ROOT / ".git").is_dir():
        if _truthy("ML_MEM_SKIP_PULL"):
            print(f"Skipping git update at {COLAB_ROOT} (ML_MEM_SKIP_PULL=1)")
        else:
            print(f"Updating clone at {COLAB_ROOT} to origin/{REPO_REF}")
            _run(["git", "-C", str(COLAB_ROOT), "fetch", "--depth", "1", "origin", REPO_REF])
            _run(["git", "-C", str(COLAB_ROOT), "reset", "--hard", "FETCH_HEAD"])
    elif not COLAB_ROOT.exists():
        _run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)])
    os.chdir(COLAB_ROOT)
    root = _find_repo_root(Path.cwd())
else:
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate the ML-MEM-Review repo (need config.yaml + src/). "
        "On Colab: re-run this cell or set ML_MEM_REPO_URL to your fork. "
        "Locally: open the notebook from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}  git={_git_head(root)[:12]}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = _truthy("ML_MEM_FORCE_INSTALL")
    head = _git_head(root)
    extras = EXTRAS
    pip_spec = str(root) if not extras else f"{root}[{extras}]"
    install_key = f"{head}\n{extras}"
    prev = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""
    if force or prev != install_key:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", pip_spec])
        try:
            marker.write_text(install_key + "\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (clone SHA + extras unchanged). Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on sys.path (no auto pip install).")

if EXTRAS:
    try:
        import ax  # noqa: F401
    except ImportError:
        pip_spec = f"{root}[{EXTRAS}]"
        print(f"Ax not found in this kernel; installing {pip_spec}")
        _run([sys.executable, "-m", "pip", "install", "-e", pip_spec])

print("Next: run the Setup import cell below.")
# === end bootstrap ===========================================================


## Setup (before the nine steps)

Load **`config.yaml`**. This file is the control panel for the notebook, and it is also the written record of the Step 1 goal. The setup cell sets a random seed so the later train/test split can be repeated.

> **Next cell:** print the dataset, lookup, `models/`, and `outputs/figures/` paths. On Colab they sit under `/content/ML-MEM-Review/`.

### Why it matters

Later cells **read this file**. They should not hide the Excel file name inside the Python. To reuse the package on *your* table, edit the YAML file, not this Python. The same engine is:

```bash
python -m run_pipeline --config config.yaml
```

### New words

| Term | Meaning |
|------|---------|
| YAML | A plain-text settings file. Here it is `config.yaml`. |
| Random seed | A starting number that makes a "random" split happen the same way again (`modeling.random_state`). |
| Kernel | The Python session behind this notebook. Packages must be installed in this session. |

### YAML

```yaml
data:
  dataset: "TutorialDataSet.xlsx"   # file inside data/raw/
  targets:                          # columns to predict
    - "Pure_Water_Permeability_..."
    - "solute1_Rejection"
```

---

**Watch out.** On Colab, run bootstrap before this cell. A warning that Ax is missing is acceptable until Step 7.

**Try.** Does the Excel path come from Python you typed, or from `config.yaml`?


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
except Exception as exc:
    Client = None
    RangeParameterConfig = None
    ChoiceParameterConfig = None
    warnings.warn(
        f"Ax import skipped during setup: {exc}. "
        "You can still run Steps 2-6; install pip install -e '.[opt]' before Step 7.",
        RuntimeWarning,
    )

PROJECT_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(PROJECT_ROOT, "config.yaml")):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not find config.yaml above the current folder. "
            "Launch the notebook from inside the repository."
        )
    PROJECT_ROOT = parent
CODE_PATH = os.path.join(PROJECT_ROOT, "src")
if CODE_PATH not in sys.path:
    sys.path.insert(0, CODE_PATH)

from utl_config_loader import (
    get_data_paths,
    get_figures_dir,
    get_model_dir,
    load_config,
    resolve_targets,
)

config = load_config(project_root=PROJECT_ROOT)
print("Configuration loaded from:", config["_config_path"])
print("Same engine as: python -m run_pipeline --config config.yaml")

plotting_cfg = config["plotting"]
FONT_SIZE = plotting_cfg["font_size"]
FIG_SIZE = tuple(plotting_cfg["global_figure_size"])
DPI = plotting_cfg["dpi"]
plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["figure.figsize"] = FIG_SIZE
plt.rcParams["savefig.dpi"] = DPI

RANDOM_STATE = config["modeling"]["random_state"]
np.random.seed(RANDOM_STATE)

data_paths = get_data_paths(config)
DATA_PATH = data_paths["dataset"]
lookup_by_align_column = data_paths.get("lookup_by_align_column", {})
CATION_PROP_PATH = lookup_by_align_column.get("S1_Cation")
ANION_PROP_PATH = lookup_by_align_column.get("S1_Anion")
if CATION_PROP_PATH is None or ANION_PROP_PATH is None:
    raise KeyError(
        "config.yaml must define feature_engineering.lookups for S1_Cation and S1_Anion."
    )

MODEL_PATH = get_model_dir(config)
FIGURES_DIR = get_figures_dir(config)
os.makedirs(FIGURES_DIR, exist_ok=True)
targets = resolve_targets(config)

print("Dataset:", os.path.basename(DATA_PATH))
print("Targets:", targets)
print("Cation lookup:", os.path.basename(CATION_PROP_PATH))
print("Anion lookup:", os.path.basename(ANION_PROP_PATH))
print("Models:", MODEL_PATH)
print("Figures:", FIGURES_DIR)


## Step 1. Set the goal and the limits (discussion)

No model is fit in this step. First turn a practical need into numbers you can optimize, and into limits the laboratory can actually meet.

**What you are deciding**

- Which results matter. For a nanofiltration membrane used to recover a resource, that might be fast water flow, strong salt rejection, and the ability to tell two dissolved chemicals apart.
- Which way to push each number. Here, higher permeability and higher rejection are better.
- What you will not change: a material you do not have, a solvent you will not use, or a reaction time your equipment cannot hold.

**What you start with:** the separation job, and what the lab can do.

**What you hand on:** a written problem statement. In this notebook that statement is already filled in as `config.yaml`. Later steps only read it. They do not invent the goal.

```yaml
data:
  targets:
    - "Pure_Water_Permeability_{Lm^{-2}^h{-1}bar^{-1}}"
    - "solute1_Rejection"
optimization:
  mode: multiobjective          # improve both numbers at once
  search_space: ...             # settings you are willing to change
  constant_parameters: ...      # settings you refuse to change
```

This tutorial searches for a nanofiltration recipe with better pure-water permeability (how fast pure water crosses the film) and better salt rejection (how much salt the film holds back).

No extra code cell. Setup already loaded this file.

**Try.** In one sentence, what would "a better membrane" mean for a project you care about? Which limit would you refuse to break?


## Step 2. Design the table, then load it

The goal of this step is a table in which **one row is one membrane experiment**: one film, tested with one feed, under one set of conditions. Each row should point back to a paper, a notebook page, or a file, so someone else can check the number.

This notebook does **not** build that table or decide which papers belong in it. The Excel file is already organized. The next cell only loads it. That loaded table is the input to Steps 3-7.

> **Next cell:** `df.shape` and `df.head()`.

### Why the column type matters

Before you model the table, separate columns by **when you would know them** if you were designing a new film. This decision keeps a later recipe search honest.

| Kind of column | When you know it | Can a recipe search change it? | Examples |
|---|---|---|---|
| **Controllable setting** | Before or during synthesis | Yes. These are the settings a search may recommend. | Monomer name and amount, solvent, additive loading, reaction time, curing temperature |
| **Fixed descriptor** | Once a chemical, support, or solute is chosen | No. It changes only if you pick a different material. | Molar mass, charge, ion size, solvent parameters |
| **After-synthesis measurement** | Only after the film exists | No. Do not use it as an input when designing a new recipe. | Thickness, roughness, contact angle, porosity |
| **Test condition and result** | During the performance test | The result is what you predict. The conditions must match the protocol. | Pressure, feed, pH, permeability, rejection |

A suggested ion size does **not** mean "set the radius to 0.3 nm on a dial." It means "choose an ion already in the table that has about that size." Thickness and roughness are results, not knobs.

Some studies later train a second model to *predict* thickness from the recipe, then use that prediction. This notebook does not do that. It simply does not treat after-synthesis measurements as settings you can choose.

### New words

| Role | Meaning | Examples in this file |
|------|---------|----------|
| **Feature** (input) | Something known before the prediction | monomer name, mass, time, `S1_Cation` |
| **Target** (output) | The number you want to predict | water permeability, solute rejection |
| **KPI** | A target that defines success for the application | how fast water crosses; how well salt is rejected |

The bundled table is about **278 rows by 36 columns**. A different file in `config.yaml` will show different numbers. Long names containing `{Lm^{-2}...}` keep the **units inside the column name**.

### How to read the output

Look for `DOI` (the paper identifier), `Dope1_Monomer1_Name`, and `solute1_Rejection`.

---

**Try.** From `df.head()`, write down one controllable setting, one fixed descriptor, and one target.


In [ ]:
# Load the dataset into a pandas DataFrame
df = pd.read_excel(DATA_PATH)

# Show the shape and first few rows of the dataset
print("Dataset shape (rows, columns):", df.shape)
df.head()

### How the columns are grouped

1. **How the film was made** — monomers, additives, solvents, and time (`Dope1_...`, `Dope2_...`). Most of these are controllable settings.
2. **How it was tested** — pressure, feed, and compaction, when the paper reported them.
3. **What was in the water** — `Solute1_Name`, `S1_Cation`, `S1_Anion`. The names are choices. Size and charge are fixed descriptors. Step 4 attaches those numbers to the names.

This cell is still only looking. Step 3 decides which rows are fit to model.


## Step 3. Inspect the table

Rows collected from different papers are not equally complete. Before any model is fit, look for blank cells, copied rows, and odd values, and ask why a number is missing.

**What you are deciding:** which rows to correct, drop, or keep. Incomplete rows can shrink the table or bias it. Duplicate experiments can make one recipe look more common than it is, and can make the score look too good.

**What you hand on:** a cleaner table, and a short note of the rule you used. "Drop every row with any blank" can erase a small materials table. That is usually the wrong rule.

> **Next cell:** `df.info()` and a duplicate count. A later cell drops exact duplicate rows if any exist. It does **not** fill blanks.

### New words

| Term | Meaning |
|------|---------|
| Missing value | A blank cell. pandas shows `NaN`, short for "not a number." |
| Duplicate row | The same experiment copied twice. Copies can inflate a score. |
| Imputation | Filling blanks. This notebook fills them **later**, inside training, using the **training rows only**. |
| Leakage | Letting the hidden test rows influence how the model is built. Filling blanks from the whole table is one way to leak. |

---

**Watch out.** A blank is not always an accident. It may mean the paper never measured that quantity.

**Try.** If reaction time is missing in 80% of rows, which is more honest: fill it with an average, drop the column, or keep an "is missing?" flag?


In [ ]:
# Check the data types and missing values
print("\nData types and missing values:")
print("---"*30)
print(df.info())

# Check for duplicates
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {num_duplicates}")

### Optional deeper dive: why values are missing

Skip this if you want. The next code cell still does not fill blanks.

| Pattern | Plain meaning | A careful response |
|---------|---------------|--------------------|
| Missing completely at random | The blank is an accident. | A simple fill can be acceptable if only a few cells are blank. |
| Missing at random | Whether a cell is blank depends on **other columns you do have**. | A fill that uses those other columns can be reasonable, but only inside the training split. |
| Missing not at random | The value is blank *because of what it would have been*. Failed films, for example, may never have been published. | Do not pretend an average repairs this. |
| Never measured | The protocol did not include it. | Drop the column. An average would invent a measurement. |

Step 5 fills holes from the **training rows only**, then applies that same rule to the test rows.


In [ ]:
# Implement duplicate-row handling identified above.
num_duplicates = df.duplicated().sum()
if num_duplicates > 0:
    print(f"Dropping {num_duplicates} duplicate rows (keeping the first occurrence).")
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    print(f"Rows remaining after deduplication: {len(df)}")
else:
    print("No duplicate rows to drop.")


In [ ]:
from utl_config_loader import resolve_features_to_model, resolve_targets

targets = resolve_targets(config)
print("data.targets in config.yaml:")
for name in targets:
    print(" -", name)
    if name not in df.columns:
        raise ValueError(f"Target {name!r} is not a column in the loaded table.")

print("\nfeatures.include lists model inputs. Ion-size columns (S1_cat_*, S1_an_*)")
print("are created in Step 4a; they are not on the raw table yet.")
print("Configured feature names:")
for name in config.get("features", {}).get("include") or []:
    print(" -", name)


## Step 4. Add useful columns and look at the data

After you have checked the rows, add columns that describe the chemistry, then look at the data before training. These two jobs repeat. A plot may tell you to drop a useless column or add a better one.

This notebook adds ion size and charge, including the Stokes radius, to the solute names. The same idea could add polymer or solvent properties from a table. Other studies sometimes turn microscope images or spectra into numbers and join those too. Image and spectrum models are **not** in this notebook.

> **Next cell:** `feature_engineering(...)` builds `df_enriched`. The printed feature list should include `S1_cat_*` and `S1_an_*`.

### Why it matters

A name like `Na+` is only text. Water and salt transport care about **size** and **charge**. The lookup files are small dictionaries: ion name to numbers. Joining them is **not** training, and it is **not** a simulator.

### YAML

```yaml
feature_engineering:
  lookups:
    - align_column: "S1_Cation"    # column in the main table
      lookup_key: "Cation"         # matching column in the ion file
      lookup_path: "data/raw/cation_prop.xlsx"
```

The ion columns are listed in `features.include`, but they do **not** exist on the raw table until this step.

---

**Watch out.** If an ion name does not match the lookup file, that row will not receive the extra numbers.

**Try.** After the cell, find one column that was **not** in Step 2's `df.head()`.


In [ ]:
from utl_feateng import feature_engineering
from utl_config_loader import resolve_features_to_model

df_enriched = feature_engineering(df, config=config, verbose=True)
features = resolve_features_to_model(config, df_enriched)
print(f"\n{len(features)} model features after enrichment (from config.yaml):")
for name in features:
    print(" -", name)
df_enriched


### The enriched table

`df_enriched` is the original recipe columns **plus** the ion descriptors. This tutorial adds only a small ion set. You could add solvent or polymer properties in the same way, through `config.yaml`.

For solutes you care about, `S1_Cation` and `S1_Anion` should now be real numbers, not blanks.


### Optional: columns that never change

**Variance** = how much a column wiggles. A constant temperature cannot help a prediction, but after **one-hot encoding** (names → 0/1 columns, Step 5) it can add clutter.

> **Next cell:** *report* near-constant columns. Modeling still uses `features.include` (same as `run_pipeline`).

Empty printed list = nothing was constant enough to flag.


In [ ]:
from utl_feateng import remove_low_variance_columns

variance_threshold = config.get("features", {}).get("low_variance_threshold", 1e-9)
_, dropped = remove_low_variance_columns(
    df_enriched,
    exclude_columns=targets,
    variance_threshold=variance_threshold,
)
print("Variance threshold:", variance_threshold)
print("Near-constant columns (not dropped for modeling; train_model uses config features):", dropped)


### Which columns the model may use, and which the search may change

`features.include` is the allow-list of **model inputs** after the ion columns are added. That list can contain both controllable settings and fixed descriptors.

Step 7 is stricter. The search may change only controllable settings, such as monomer, amount, and time. Fixed descriptors stay tied to the chemical you chose. Measurements that exist only after synthesis, such as thickness or roughness, must not be search knobs. If they were, the search could "recommend" a thickness you cannot dial in when you mix the recipe.

The output of this step is a modeling-ready table: validated inputs, the performance targets, and enough labels to repeat the work. Plots and training both use `df_enriched` and the same YAML file as `python -m run_pipeline`.


### 4b. Look at the data before training

Check how the targets are spread, and which numbers rise and fall together. This is a quality check, not the scientific conclusion.

> **Next cell:** `target_distributions.png` and `pearson_heatmap.png` under `outputs/figures/`.

### Why it matters

If permeability sits in a tiny band, a "good" score can be cheap: always guess the average. If two numeric columns are copies of each other, the model may use either one. Membrane tables are often small and mixed together from many papers, so this check matters. A tidy correlation can be an accident of which experiments were published.

### New words

| Term | Meaning |
|------|---------|
| Pearson correlation | A number from -1 to 1 for a *straight-line* link between two **numeric** columns. It ignores monomer names. |

Near 1, the two columns rise together. Near -1, one rises as the other falls. Near 0, there is no straight-line link. The heatmap hides pairs weaker than the YAML cutoff (here **0.5**).

### How to read the output

- **Distribution plot:** spread, skew, and unusual rows. An unusual row is not automatically an error.
- **Heatmap:** a strong correlation is not proof that one setting causes the other.

---

**Watch out.** Do not drop a lab setting only because its Pearson correlation is small. Tree models can still use curved effects and combinations of settings.

**Try.** Will the monomer name appear on the heatmap? (No. It is a name, not a number.)


In [ ]:
from utl_eda import plot_target_distributions, compute_and_plot_pearson_heatmap
from utl_fileIO import file_io

fig, _ = plot_target_distributions(df_enriched, target_columns=targets, config=config)
file_io("save", "figure", FIGURES_DIR, "target_distributions.png", data=fig)
print("Saved target_distributions.png")

numeric_df = df_enriched.select_dtypes(include=["number"])
threshold = (config.get("data_analysis", {}) or {}).get("correlation_thresholds", [0.5])[-1]
_, heatmap_fig = compute_and_plot_pearson_heatmap(
    numeric_df, threshold=threshold, config=config
)
file_io("save", "figure", FIGURES_DIR, "pearson_heatmap.png", data=heatmap_fig)
print("Saved pearson_heatmap.png")


### Reading the heatmap

A strong correlation can mean "papers tend to report these two settings together," not "A causes B." A weak Pearson value does not mean "useless." Step 5 uses trees, so combinations such as monomer, time, and additive can still matter.

If two numeric columns are almost the same measurement, keep the one you can set in the lab. That choice also decides what Step 7 is allowed to search.

If a plot shows a column that never changes, two copied columns, or a target with almost no spread, go back and edit the feature list before training. Adding columns and plotting them is a loop, not two unrelated chores.


## Step 5. Fit a model and test it on hidden rows

Once the feature list is fixed, hold back a test set **before** you choose or tune a model. This notebook holds back about 20% of the rows. A fixed random seed makes the same split happen again if you re-run the cell.

Training then uses one pipeline: fill blanks, scale numbers, turn names into 0/1 columns, and fit the model. Those cleaning steps are learned from the **training rows only**, which limits leakage. Choices among models and settings are compared inside the training set. The hidden test rows are used at the end.

> **Next cell:** train and test metrics, `parity_<target>.png`, and saved models in `models/`. `force_retrain=True` means edits to `config.yaml` are picked up.

### Why it matters

- **Train** (about 80%): the model may learn from these rows.
- **Test** (about 20%): held back. If the model also does well here, it is not only memorizing the homework.

A great training score and a poor test score means **overfitting**: the model learned this table, not a rule you can trust for a new recipe.

If several rows come from the same paper or the same membrane batch, a random split can put near-copies in both sides and make the test score look too good. A grouped split would keep each paper on only one side. This notebook uses a random split, so treat a very high test score with that caution.

### New words

| Term | Meaning |
|------|---------|
| ExtraTrees | A crowd of decision trees. Each tree asks yes/no questions, such as "was the reaction longer than 5 minutes?" The model averages many slightly random trees. It is a flexible fitter for tables, not a chemistry theory. |
| One-hot encoding | A category name becomes several 0/1 columns, such as PIP = 1 and PEI = 0. The list of names is learned from the **training rows only**. |
| R² | Often between 0 and 1, and it can be negative. 1 means perfect. 0 means no better than guessing the average. Compare the training score with the test score. |
| RMSE and MAE | Typical error in the **same units as the target**. Smaller is better. RMSE punishes large misses more than MAE. |

### YAML

```yaml
modeling:
  model_selection: extratrees   # or randomforest, ridge, ...
  task: regression
  train_test_ratio: 0.2
```

Change `model_selection` in the YAML file, not by editing this cell. Tree models suit this table. Microscope images would need a different model. This notebook does not train image models.

### How to read the output

Points near the diagonal mean the prediction is close to the measurement. A long target name in the plot title is normal. The cell also prints R², RMSE, and MAE.

What you hand to Step 6 is the fitted model, those test numbers, and the parity plot. For a small membrane table, also ask whether the features cover the recipes you hope to make, and how noisy the experiments are. A pretty score on a narrow table is not a license to search far outside it.

---

**Watch out.** Do not trust the training R² alone.

**Try.** Suppose the training R² is 0.99 and the test R² is 0.2. Would you trust the recipes from Step 7? Why or why not?


In [ ]:
from utl_fileIO import file_io
from utl_modelling import _safe_target, evaluate_and_plot_results, train_model

for target in targets:
    print(f"\n=== Modeling target: {target} ===")
    pipe, X_train, X_test, y_train, y_test = train_model(
        df=df_enriched,
        target=target,
        config_path=config["_config_path"],
        force_retrain=True,
    )
    fig, resolved_task = evaluate_and_plot_results(
        target=target,
        y_train=y_train,
        y_pred_train=pipe.predict(X_train),
        y_test=y_test,
        y_pred_test=pipe.predict(X_test),
        config=config,
        fig_size=FIG_SIZE,
    )
    plot_name = (
        f"confusion_{_safe_target(target)}.png"
        if resolved_task == "classification"
        else f"parity_{_safe_target(target)}.png"
    )
    file_io("save", "figure", FIGURES_DIR, plot_name, data=fig)
    print("Saved", plot_name)


### How to read a parity plot

Each point is one experiment: **measured** vs **predicted**.

- **Dashed diagonal:** prediction = measurement.
- **Above / below** the line: over- / under-predicted.
- Train tight, test scattered → overfitting.
- Bend at high permeability → weak there; inverse-design suggestions in that region are shaky.

Classification uses a **confusion matrix** instead. This tutorial’s KPIs are numbers → parity.

**Try.** Where would “always under-predicts high rejection” show up?


## Step 6. Check what drove the predictions (SHAP)

A test score says whether the predictions match. It does not say why. This step takes the fitted model and asks which input columns pushed a prediction up or down.

Use that answer as a quality check **before** any recipe search. Carry a model forward only if the important columns make chemical sense and agree with how these films are known to behave. If the top driver is a paper ID, a column that never changes, or something that contradicts the process, do not search yet. Go back to Step 4 and fix the feature list.

An unexpected driver can also be useful. It may show a recording error, or it may suggest one targeted experiment. It should not steer the search until you understand it.

> **Next cell:** `shap_summary_*.png` and `shap_importance_*.png`. If the `shap` package is missing, the cell tries to install it, or skips so you can still read Step 7. A skipped check is not the same as a passed check.

### Why it matters

SHAP explains **this fitted model**. It is not proof of membrane physics, and it is not the same thing as causation. You do not need the game-theory derivation to read the plots.

### New words

| Term | Meaning |
|------|---------|
| SHAP | Short for SHapley Additive exPlanations. It splits one prediction into a contribution from each input. In the summary plot, top rows matter more on average. Color shows a high or low feature value. Left or right shows whether that value pushed the prediction down or up. |

If there are many columns, only the **top 20** are drawn. Names such as `Dope1_Monomer1_Name_PIP` are the 0/1 columns from Step 5. That is expected.

---

**Watch out.** Do not write "SHAP says additive loading causes rejection." Safer: "In this model, additive loading was associated with higher rejection."

**Try.** Circle one setting you can change in the lab. That is a candidate for the Step 7 search. A fixed descriptor, such as ion charge, may explain the model, but it is not a dial you turn.


### Beeswarm vs bar

- **Beeswarm (summary):** each dot is one sample — use for **direction**.
- **Bar (importance):** mean \|SHAP\| — a **ranking** without direction.

Names like `Dope1_Monomer1_Name_PIP` are one-hot columns from Step 5. That is expected.


In [ ]:
try:
    import shap  # noqa: F401
except ImportError:
    import subprocess
    spec = f"{PROJECT_ROOT}[shap]"
    print("SHAP not found; installing", spec)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    import shap  # noqa: F401

from utl_config_loader import resolve_features_to_model
from utl_fileIO import file_io
from utl_modelling import _safe_target
from utl_shap import shap_explain_target

features = resolve_features_to_model(config, df_enriched)
for target in targets:
    try:
        shap_values, feature_names, summary_fig, bar_fig = shap_explain_target(
            target=target,
            df_enriched=df_enriched,
            features_to_model=features,
            config=config,
            max_samples=1000,
        )
    except ImportError as exc:
        print(exc)
        print("Skipping SHAP. You can continue with Step 7.")
        break
    if summary_fig is None or bar_fig is None:
        print("SHAP skipped for", target)
        continue
    safe = _safe_target(target)
    file_io("save", "figure", FIGURES_DIR, f"shap_summary_{safe}.png", data=summary_fig)
    file_io("save", "figure", FIGURES_DIR, f"shap_importance_{safe}.png", data=bar_fig)


### From the plots to a decision

If monomer, additive, time, or ion size ranks high, and the direction matches what you know about making these films, the model is at least consistent with the chemistry. If the top feature is a paper ID, or a column that never changes, the feature list is wrong.

Step 7 should vary only a short list of settings you can actually set. If that list fights both the SHAP plot and common sense, edit `config.yaml` before you trust the search. The result of this step is either an accepted model, ready to act as a stand-in experiment, or a rejected one that goes back to Step 4.


## Step 7. Search for recipes

Once Step 6 has accepted the model, two jobs are possible.

- **Screening:** score many recipes you already wrote down, and keep the promising ones.
- **Inverse design:** start from the performance you want, and search for settings the model predicts will get you there.

This notebook does the second job. ExtraTrees stands in for "make the film and measure it." **Ax** then searches recipes that trade off permeability and rejection. The same idea can later recommend operating settings as conditions change. This tutorial searches fabrication recipes.

The search must stay inside the limits from Step 1, and it may change only the controllable settings from Step 2. A suggested ion size means "pick an ion in the lookup table with about that size," not "set the radius on a dial." A suggested concentration, time, or temperature must be a value your equipment can actually deliver. Thickness, roughness, and contact angle are predicted outcomes, not settings to manipulate.

> **Next cells:** build the search from the YAML file, run the trials, print the recipes, then draw the figures. The first Ax install can take several minutes (`pip install -e ".[opt]"`).

### Why it matters

Inverse design means we name the performance we want and search for the inputs. The optimizer never makes a real film. It only asks the model.

**Analogy.** ExtraTrees is a flight simulator. The laboratory is the aircraft. Keep a shortlist from the simulator. You still have to fly the plane. That flight is Step 8.

This tutorial uses `optimization.mode: multiobjective`. You get a set of trade-off recipes, not one weighted winner. The next subsection explains that set.

### New words

| Term | Meaning |
|------|---------|
| Ax | A free Python library for Bayesian optimization. |
| Bayesian optimization | A search that proposes the next recipe using a model of where good results might be. It mixes untried regions (exploration) with regions that already look good (exploitation). |

### YAML

```yaml
optimization:
  mode: multiobjective
  n_trials: 100          # more trials are slower, and often give a better shortlist
  search_space: ...      # settings Ax may change
  constant_parameters:   # settings held fixed
```

### How to read the output

You should see completed trials, printed recipes, and `inverse_design_summary.png`. On Colab, the figures are under `/content/ML-MEM-Review/outputs/figures/`.

Read each printed recipe as a lab instruction: material names, amounts, and fabrication settings, plus the measurements to bring back in Step 8 using the same test as the training table. The notebook prints that shortlist and stops. It does not declare a recipe successful.

---

**Watch out.** This search is not for pass/fail labels. You need the model files from Step 5. After changing `mode`, delete `models/ax_client.pkl`, or keep `force_rebuild=True`.

**Try.** On a trade-off plot, what would high permeability and high rejection mean, and why might no trial sit in that corner?


### What the search may change

| | Meaning |
|--|---------|
| **Search space** | Settings Ax may change: a range, such as time or mass, or a choice, such as which monomer. These must be controllable in the lab. Stored in `optimization.search_space`. |
| **Constants** | Inputs you refuse to vary, such as the solvent or the test salt. Stored in `constant_parameters`. Fixed descriptors stay with the chemical you picked. |

**Hard rule.** The columns Ax can change, plus the columns held constant, must together match the inputs the model was trained on. Keep the search small so the suggestions stay plausible.

A suggested Stokes radius is not a knob. It means choose a different ion or monomer that the lookup table already describes. Do not let the search set thickness, roughness, or contact angle.

> **Next cell:** how many Ax parameters were built. Check that list against what your lab can actually change.


In [ ]:
if Client is None or RangeParameterConfig is None or ChoiceParameterConfig is None:
    import subprocess
    spec = f"{PROJECT_ROOT}[opt]"
    print("Ax not found; installing", spec)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig

from utl_Ax import (
    build_ax_search_space,
    get_optimization_recommendations,
    oracle_from_params,
    plot_ax_results,
    results_to_dataframe,
    run_ax_optimization,
)


In [ ]:
opt_cfg = config["optimization"]
search_space_parameters = build_ax_search_space(
    opt_cfg=opt_cfg,
    choice_param_cls=ChoiceParameterConfig,
    range_param_cls=RangeParameterConfig,
)
print("optimization.mode:", opt_cfg.get("mode"))
print("Search parameters from config.yaml:", len(search_space_parameters))
for item in search_space_parameters:
    print(" -", getattr(item, "name", item))


### The stand-in experiment

Each Ax trial asks the saved models instead of the laboratory. The helper does four things:

1. Merge the proposed settings with `constant_parameters`.
2. Build a one-row table and add ion descriptors in the **same** way as Step 4.
3. Predict each target with the saved ExtraTrees pipelines.
4. Hand those numbers back to Ax.

You do not call this helper yourself. `run_ax_optimization` does. The numbers are still model guesses. Step 8 is the real measurement.

---

**Watch out.** If `models/*.pkl` is missing, go back to Step 5. If a YAML name is not a training feature, the row cannot be scored.


### Two goals: a Pareto front, not one score

Permeability and rejection **compete**. There is a **set** of good recipes, not a single winner.

> **Next cell:** prints `mode` and each objective’s direction and threshold.

### New words

| Term | Meaning |
|------|---------|
| Dominated | Another recipe is better on **both** KPIs. |
| Pareto frontier | Recipes that are *not* dominated: you cannot improve one KPI without hurting the other. |
| Threshold (YAML) | Soft constraint on the **normalized** KPI (scaled ~0–1 with `normalization_min` / `max`). |

### How to read the maps

- **Blue triangles** = Pareto. **Red circles** = all trials.

### Optional deeper dive

`mode: scalarized` + `scalarized_objective.equation` collapses two KPIs into **one weighted number** (example 05). Faster, but “best” is only best *for those weights*.


In [ ]:
opt_cfg = config["optimization"]
print("mode:", opt_cfg.get("mode"))
for obj in opt_cfg.get("objectives") or []:
    print(
        f"  - {obj.get('name')}: {obj.get('direction')} "
        f"threshold={obj.get('threshold')} "
        f"-> {obj.get('model_target') or obj.get('original_name')}"
    )
eq = (opt_cfg.get("scalarized_objective") or {}).get("equation")
if opt_cfg.get("mode") == "scalarized":
    print("scalarized_objective.equation:", eq)


### The optimization loop

A few structured-random (**Sobol**) starts, then Bayesian trials. `n_trials` is the budget. Large values (the tutorial file uses 100) can take **many minutes**.

> **Next cell:** `run_ax_optimization`. Checkpoint under `models/` (`client_cache_file`).

**Sobol** = diverse random starts, better than throwing darts blindly. Re-running without `force_rebuild=True` may **reload** the checkpoint instead of searching again.

Treat the result as a **shortlist of experiments**, not a new membrane.

---

**Watch out.** No Ax in this kernel → bootstrap or `pip install -e ".[opt]"`.


In [ ]:
client, results = run_ax_optimization(
    search_space_parameters=search_space_parameters,
    oracle_from_params=oracle_from_params,
    config=config,
    sobol_init=opt_cfg["sobol_init"],
    n_trials=opt_cfg["n_trials"],
    force_rebuild=True,
    random_seed=config.get("modeling", {}).get("random_state"),
    save_every_trials=opt_cfg["save_checkpoint_every"],
    client_path=os.path.join(MODEL_PATH, opt_cfg["client_cache_file"]),
)
print("Optimization completed:", len(results), "trials")


### Reading the suggested recipes

> **Next cell:** `get_optimization_recommendations` prints a set of trade-off recipes for this search box and this ExtraTrees model.

Read each row as a lab instruction, not as a prize.

1. **Materials:** names you can order, or already have.
2. **Amounts:** masses or volumes a balance or pipette can deliver.
3. **Conditions:** times and temperatures your equipment can hold.
4. **What to measure next:** permeability and rejection, with the same protocol as the training table.

A negative mass, a monomer you cannot obtain, or a time of a fraction of a second means the model, the bounds, or the constants are wrong. Do not "fix" that recipe on the bench. Fix the search setup and run it again.

The notebook stops at this printed shortlist. Whether the prediction was right is a laboratory question.

---

**Try.** Would you make the top recipe this week? Which measurement would make you trust the model, and which result would send the row back into the table?


In [ ]:
recommendations = get_optimization_recommendations(
    client=client,
    optimization_mode=opt_cfg["mode"],
    top_k=5,
)


### Pictures of the search

> **Next cell:** `inverse_design_summary.png` (metrics vs trial — 3D “waterfall”) and **response maps**.

| Figure | How to read it |
|--------|----------------|
| Waterfall | Each ribbon is one quantity vs trial index (0, 1, 2, …). You want movement after the random start, not a flat line. |
| Response map | Smoothed landscape vs two knobs from **these few trials** — a cartoon, not nature. **Contour** = 2D; **surface** = 3D of the **same** interpolation. |
| Symbols | Red circles = all trials. Blue triangles = Pareto (if `mode: multiobjective`). |

If a map is skipped (too few distinct points), the summary figure is still enough. Files: figures directory from Setup.


In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
from utl_fileIO import file_io

trial_df = results_to_dataframe(results)
viz = plot_ax_results(trial_df=trial_df, config=config, client=client)

summary_figure = (
    viz.get("summary", {}).get("figure") if isinstance(viz.get("summary"), dict) else None
)
if summary_figure is None and viz.get("figures"):
    summary_figure = viz["figures"].get("summary_figure")
if summary_figure is not None:
    file_io("save", "figure", FIGURES_DIR, "inverse_design_summary.png", data=summary_figure)

for index, response_map in enumerate(viz.get("response_maps", []), start=1):
    if response_map.get("contour_figure") is not None:
        file_io(
            "save", "figure", FIGURES_DIR,
            f"response_map_{index}_contour.png",
            data=response_map["contour_figure"],
        )
    if response_map.get("surface_figure") is not None:
        file_io(
            "save", "figure", FIGURES_DIR,
            f"response_map_{index}_surface.png",
            data=response_map["surface_figure"],
        )

plt.close("all")
for name in [
    "inverse_design_summary.png",
    "response_map_1_contour.png",
    "response_map_1_surface.png",
    "response_map_2_contour.png",
    "response_map_2_surface.png",
]:
    path = Path(FIGURES_DIR) / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))


## Step 8. Test the shortlist in the lab (discussion)

Steps 5-7 only asked a model. This step asks the laboratory. There is no code cell here on purpose. This repository does not run a coating machine, a test cell, or a robot.

**What you are deciding:** whether the measured result is close enough to stop, or whether the new rows must go back into the table.

**What to do**

1. From the Step 7 shortlist, pick one to three recipes that look chemically sensible. A high model score is not enough.
2. Make the films with the materials, amounts, and conditions on the printout.
3. Measure pure-water permeability and salt rejection with the **same protocol** as `TutorialDataSet.xlsx`. If you need to check that the film is the structure you intended, microscopy or spectroscopy can help. Those checks support the performance test. They are not a second optimization, and this notebook does not analyze those images.
4. Write the new rows back into the table, using the same columns as Step 2. Keep **successes and failures**. If you hide the failures, the next model never learns the dead ends.

If the measurements agree with the prediction, this cycle can stop. If they do not, send the recipes and the measured numbers to Step 9.


## Step 9. Add the new results and search again (discussion)

A printed recipe is a candidate, not a finished membrane. The measured permeability or rejection can miss the prediction when the new experiment sits outside the old table, when the film varies from batch to batch, when the measurement is noisy, or when an important descriptor was never recorded.

**What you hand on:** new rows in the same table, then a refitted model and a new shortlist.

In this package, closing the loop means:

1. Add the Step 8 rows to the Excel or CSV file. Do not change what the columns mean.
2. Re-run Steps 3-7. Set `force_retrain=True`, and delete or rebuild `models/ax_client.pkl`, so you are not looking at the old search.
3. Compare the new shortlist with the previous one.

The code stops at Step 7 so the same workflow can run on any table, including one with no laboratory attached. Steps 8 and 9 are the experiment and the update, not extra Python.

Later cycles can be more selective than "add every row and refit."

- Test first where the model is most unsure. That idea is called **active learning**.
- Reject a suggestion that breaks a known physical limit, such as a water flow faster than the pressure and thickness allow. That is a **physics check**.
- If you move to a new membrane family and have only a few new rows, start from a model trained on a larger related table and adjust it. That is **transfer learning**.

Those are options for a later project. This notebook does not run them. The required update is simpler: add the measured rows, refit, and search again.


## Recap, troubleshooting, next steps

The paper loop has Steps 1-9. This notebook discusses all nine. The modeling code you run is Steps 4-7. Steps 1-3 are your decisions; the notebook only loads and inspects a prepared table. Steps 8-9 are laboratory work. They are discussed here and are not automated.

### Key concepts

- Write the goal and the laboratory limits before fitting anything.
- A **controllable setting** can be recommended. A **fixed descriptor** changes only when the chemical changes. An **after-synthesis measurement** is not a knob.
- Features are inputs. Targets are the numbers you predict. Training rows teach the model. Test rows check it. A correlation is not a cause.
- The model stands in for the experiment. SHAP checks that stand-in. Ax searches it.
- The recipe search is for numeric targets. This tutorial searches two targets at once. Example 05 collapses them into one weighted score.
- After the shortlist, test it in the lab (Step 8), add the rows, and refit (Step 9).

### Troubleshooting

| Problem | What to try |
|---------|-------------|
| `config.yaml` or `utl_*` not found | Run bootstrap, then Setup, in that order. |
| `No module named shap` | In this notebook's kernel, run `pip install -e ".[shap]"`. Or skip to Step 7 and treat the check as skipped, not passed. |
| Ax or optimize error | Run `pip install -e ".[opt]"`. Confirm that Step 5 wrote `.pkl` files in `models/`. |
| No blue triangles on the trade-off plot | Set `mode: multiobjective`. Delete `models/ax_client.pkl`. Keep `force_rebuild=True`. |
| Old figures or an old model | Set `force_retrain=True` and `force_rebuild=True`, or delete the files in `models/`. |
| The Colab copy looks old | Re-run bootstrap after the GitHub project is updated. |

### Practice (pick one)

1. Change `data.targets` to another numeric column, then re-run Steps 4 and 5.
2. Set `modeling.model_selection: randomforest` and compare the test RMSE.
3. Set `optimization.mode: scalarized` and compare that single winner with the trade-off set.

None of this replaces making the membrane. It helps you choose which membrane to make next.
